# Q3 KPI snapshot | Thursday board pack

The board request follows the Tuesday marketing decision in triage. This notebook answers the Q3 KPI request and tests the 59M EGP deposit headline against the extract. It builds on the profiling results, using the Task 2 cleaned exports. The raw transaction CSV supports alternative-scope comparisons.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data").is_dir())
DATA = ROOT / "data"
EXTRACT_DATE = pd.Timestamp("2026-11-05")
Q3_START = pd.Timestamp("2026-07-01")
Q3_END = pd.Timestamp("2026-10-01")
MONTHS = pd.period_range("2026-07", "2026-09", freq="M")
# Fixed offset documented in the case README.
CAIRO_OFFSET = pd.Timedelta(hours=2)
HEADLINE_CLAIM = 59_000_000

# QA users to be excluded from analysis. These are known test accounts and should not be counted in the analysis.

CLEANED = DATA / "cleaned"
users = pd.read_csv(CLEANED / "users.csv", parse_dates=["signup_ts", "kyc_verified_ts"])
txn = pd.read_csv(CLEANED / "transactions.csv", parse_dates=["txn_ts"])
loans = pd.read_csv(CLEANED / "loans.csv", parse_dates=["disbursed_date", "due_date", "last_repayment_date"])
plus = pd.read_csv(CLEANED / "plus_subscriptions.csv", parse_dates=["plus_start_date", "plus_end_date"])
deposit_audit = pd.read_csv(CLEANED / "q3_deposit_audit.csv")


def in_q3(timestamp):
    return timestamp.ge(Q3_START) & timestamp.lt(Q3_END)


## Deposit reconciliation

Use the README's fixed UTC+2 offset for the primary calendar and retain the UTC result as a boundary check. A deposit counts if it is successful, has a valid non-QA user, is unique, and falls in the quarter. Each bridge step measures a reproducible change. The source calculation for the claimed headline was not supplied.

In [3]:
# Raw records are used only for the alternative-scope checks.
raw_path, = (DATA / "raw").glob("* - transactions.csv")
raw_txn = pd.read_csv(raw_path, parse_dates=["txn_ts"])
raw_deposits = raw_txn.loc[raw_txn.txn_type.eq("deposit")]

txn["cairo_ts"] = txn.txn_ts + CAIRO_OFFSET
success = txn.loc[txn.status.eq("success")]
q3_success = success.loc[in_q3(success.cairo_ts)].copy()
q3_deposits = q3_success.loc[q3_success.txn_type.eq("deposit")].copy()
deposit_volume = q3_deposits.amount_egp.sum()

bridge = deposit_audit.rename(columns={"deposits": "rows"}).set_index("scope")[["rows", "volume_egp"]]
bridge.index = pd.Index([
    "Raw UTC Q3 deposits, all statuses", "Successful deposits only",
    "Remove exact copies", "Exclude QA and unmatched users",
], name="step")
bridge.loc["Use Cairo dates per README"] = [len(q3_deposits), deposit_volume]
bridge["delta_from_previous_egp"] = bridge.volume_egp.diff()
bridge["gap_to_claim_egp"] = HEADLINE_CLAIM - bridge.volume_egp
display(bridge)

wrong_scope = pd.Series({
    "Raw all-date deposit amounts, all statuses": raw_deposits.amount_egp.sum(),
    "Raw all-date successful deposit amounts": raw_deposits.loc[raw_deposits.status.eq("success"), "amount_egp"].sum(),
    "Clean all-date successful deposit amounts": success.loc[success.txn_type.eq("deposit"), "amount_egp"].sum(),
    "Raw Q3 UTC all transaction types and statuses": raw_txn.loc[in_q3(raw_txn.txn_ts), "amount_egp"].sum(),
    "Clean Q3 Cairo all successful transaction types": q3_success.amount_egp.sum(),
}, name="volume_egp")
display(wrong_scope.to_frame())

daily = q3_deposits.groupby(q3_deposits.cairo_ts.dt.normalize()).amount_egp.agg(["size", "sum"])
daily = daily.reindex(pd.date_range(Q3_START, Q3_END - pd.Timedelta(days=1)), fill_value=0)
spike_date = daily["sum"].idxmax()
spike_egp = daily.loc[spike_date, "sum"]
display(daily.nlargest(5, "sum").rename(columns={"size": "deposits", "sum": "volume_egp"}))
print(f"Q3 deposits: {deposit_volume:,.2f} EGP | {HEADLINE_CLAIM - deposit_volume:,.2f} EGP below the claim")
print(f"Largest Cairo day: {spike_date:%Y-%m-%d}, {spike_egp:,.2f} EGP ({spike_egp / deposit_volume:.1%} of Q3)")
print(f"Median day: {daily['sum'].median():,.2f} EGP")

,rows,volume_egp,delta_from_previous_egp,gap_to_claim_egp
step,,,,
"Raw UTC Q3 deposits, all statuses",9131,61044000.0,NaN,-2044000.0
Successful deposits only,7782,59477600.0,-1566400.0,-477600.0
Remove exact copies,7671,58485800.0,-991800.0,514200.0
Exclude QA and unmatched users,7001,19895800.0,-38590000.0,39104200.0
Use Cairo dates per README,6995,19893500.0,-2300.0,39106500.0


,volume_egp
"Raw all-date deposit amounts, all statuses",66902700.00
Raw all-date successful deposit amounts,64994100.00
Clean all-date successful deposit amounts,24848900.00
Raw Q3 UTC all transaction types and statuses,66180490.53
Clean Q3 Cairo all successful transaction types,24641576.70


,deposits,volume_egp
2026-09-03,89,10297000.0
2026-09-04,113,1061600.0
2026-09-29,148,252900.0
2026-09-20,124,174400.0
2026-09-25,115,172400.0


Q3 deposits: 19,893,500.00 EGP | 39,106,500.00 EGP below the claim
Largest Cairo day: 2026-09-03, 10,297,000.00 EGP (51.8% of Q3)
Median day: 94,000.00 EGP


## Signup cohorts, activation, and monthly active users

Signup cohorts follow Cairo calendar months. Activation requires KYC verified by the extract and a successful deposit between signup and 168 hours later, inclusive. The seven-day limit applies to the deposit. KYC need not precede the deposit. The quarter-end signup cohort has a full observation window in this extract. MAU counts distinct users with at least one successful transaction of any type during the Cairo month.

In [4]:
users["signup_cairo"] = users.signup_ts + CAIRO_OFFSET
cohort = users.loc[in_q3(users.signup_cairo)].copy()
cohort["month"] = cohort.signup_cairo.dt.to_period("M")

deposit_candidates = success.loc[success.txn_type.eq("deposit"), ["user_id", "txn_ts"]].merge(
    cohort[["user_id", "signup_ts", "kyc_verified_ts"]], on="user_id", validate="many_to_one"
)
delay = deposit_candidates.txn_ts - deposit_candidates.signup_ts
activated_ids = deposit_candidates.loc[
    delay.between(pd.Timedelta(0), pd.Timedelta(days=7))
    & deposit_candidates.kyc_verified_ts.lt(EXTRACT_DATE + pd.Timedelta(days=1)),
    "user_id",
].unique()
cohort["activated"] = cohort.user_id.isin(activated_ids)

monthly = cohort.groupby("month").activated.agg(signups="size", activated_users="sum").reindex(MONTHS, fill_value=0)
monthly["activation_rate_pct"] = 100 * monthly.activated_users / monthly.signups
monthly["mau"] = q3_success.groupby(q3_success.cairo_ts.dt.to_period("M")).user_id.nunique().reindex(MONTHS, fill_value=0)
monthly["deposit_only_mau"] = q3_deposits.groupby(q3_deposits.cairo_ts.dt.to_period("M")).user_id.nunique().reindex(MONTHS, fill_value=0)
monthly["deposit_volume_egp"] = q3_deposits.groupby(q3_deposits.cairo_ts.dt.to_period("M")).amount_egp.sum().reindex(MONTHS, fill_value=0)
display(monthly.round(2))

,signups,activated_users,activation_rate_pct,mau,deposit_only_mau,deposit_volume_egp
2026-07,1795,742,41.34,702,702,1276000.0
2026-08,2000,790,39.50,1391,1233,3063900.0
2026-09,2396,825,34.43,2048,1711,15553600.0


## Loan disbursements and DPD30

Use disbursal date for cohorts. A loan is observable at DPD30 when its due date plus 30 days is on or before November 5. Use principal plus the supplied fee to test full repayment. A loan already fully repaid in the extract is counted as defaulted at DPD30 if its final repayment came after the DPD30 cutoff. Treat repayment on the cutoff date as on time for this measure. The extract has no daily balance history.

In [5]:
q3_loans = loans.loc[in_q3(loans.disbursed_date)].copy()
q3_loans["month"] = q3_loans.disbursed_date.dt.to_period("M")
q3_loans["dpd30_date"] = q3_loans.due_date + pd.Timedelta(days=30)
q3_loans["mature"] = q3_loans.dpd30_date.le(EXTRACT_DATE)
full_at_extract = q3_loans.amount_repaid_egp.ge(q3_loans.principal_egp + q3_loans.fee_egp - .01)
assert q3_loans.loc[full_at_extract, "last_repayment_date"].notna().all()
full_by_dpd30 = full_at_extract & q3_loans.last_repayment_date.le(q3_loans.dpd30_date)
q3_loans["dpd30_default"] = q3_loans.mature & ~full_by_dpd30

loan_kpis = q3_loans.groupby("month").agg(
    loans_disbursed=("loan_id", "size"), principal_egp=("principal_egp", "sum"),
    mature_loans=("mature", "sum"), defaults=("dpd30_default", "sum"),
).reindex(MONTHS, fill_value=0)
loan_kpis["immature_loans"] = loan_kpis.loans_disbursed - loan_kpis.mature_loans
loan_kpis["dpd30_rate_pct"] = 100 * loan_kpis.defaults / loan_kpis.mature_loans
display(loan_kpis.round(2))

,loans_disbursed,principal_egp,mature_loans,defaults,immature_loans,dpd30_rate_pct
2026-07,260,341500,260,17,0,6.54
2026-08,427,548000,427,25,0,5.85
2026-09,623,826000,353,34,270,9.63


## Plus subscribers and consolidated snapshot

A subscriber is active on a date if their start date is on or before that date and their end date is blank or later. This treats the end date as the first inactive date. Report distinct subscriber IDs. Date-only fields are used as supplied.

In [6]:
def active_plus(as_of):
    active = plus.plus_start_date.le(as_of) & (plus.plus_end_date.isna() | plus.plus_end_date.gt(as_of))
    return plus.loc[active, "user_id"].nunique()


plus_counts = pd.Series({
    "Active at Q3 close": active_plus(Q3_END - pd.Timedelta(days=1)),
    "Started during Q3": plus.loc[in_q3(plus.plus_start_date), "user_id"].nunique(),
    "Active at extract": active_plus(EXTRACT_DATE),
}, name="subscribers")
display(plus_counts.to_frame())

n_mature = int(q3_loans.mature.sum())
n_defaults = int(q3_loans.dpd30_default.sum())
snapshot = pd.DataFrame([
    ("Q3 signups", f"{len(cohort):,}", "Distinct eligible users signed up in the Cairo quarter"),
    ("Q3 activation rate", f"{cohort.activated.mean():.2%} ({cohort.activated.sum():,}/{len(cohort):,})",
     "KYC verified at extract and successful deposit within 168 hours"),
    ("Monthly active users", " / ".join(f"{v:,}" for v in monthly.mau),
     "July / August / September, distinct users with a successful transaction"),
    ("Q3 deposit volume", f"{deposit_volume:,.2f} EGP", "Successful, unique, non-QA deposits, Cairo dates"),
    ("Q3 loans disbursed", f"{len(q3_loans):,} / {q3_loans.principal_egp.sum():,.2f} EGP",
     "Count and total principal for loans disbursed in Q3"),
    ("Q3 DPD30 default rate", f"{n_defaults / n_mature:.2%} ({n_defaults:,}/{n_mature:,})",
     "Q3 disbursals that reached DPD30 by November 5"),
    ("Q3-end active Plus subscribers", f"{plus_counts['Active at Q3 close']:,}",
     "Distinct active subscribers on September 30"),
], columns=["metric", "value", "definition"]).set_index("metric")
display(snapshot)

,subscribers
Active at Q3 close,85
Started during Q3,93
Active at extract,82


,value,definition
metric,,
Q3 signups,"6,191",Distinct eligible users signed up in the Cairo...
Q3 activation rate,"38.07% (2,357/6,191)",KYC verified at extract and successful deposit...
Monthly active users,"702 / 1,391 / 2,048","July / August / September, distinct users with..."
Q3 deposit volume,"19,893,500.00 EGP","Successful, unique, non-QA deposits, Cairo dates"
Q3 loans disbursed,"1,310 / 1,715,500.00 EGP",Count and total principal for loans disbursed ...
Q3 DPD30 default rate,"7.31% (76/1,040)",Q3 disbursals that reached DPD30 by November 5
Q3-end active Plus subscribers,85,Distinct active subscribers on September 30


## Context for the eventual board summary

- The 59M EGP headline is not supported by the Q3 extract. The validated result is **19,893,500 EGP**, **39,106,500 EGP** below the claim. Raw successful Q3 deposits total 59.48M EGP before deduplication and QA exclusion, a plausible source for the rounded claim. Removing exact copies subtracts 991,800 EGP, excluding QA subtracts 38.59M EGP, and the Cairo calendar subtracts 2,300 EGP. QA is the dominant measured driver. The original calculation remains unverified.
- The largest Cairo deposit day is September 03 at **10,297,000 EGP** (51.8% of Q3). The cluster is retained. The extract does not establish that the deposits are invalid.
- Use the consolidated snapshot for context when drafting the board communication. Keep the mature-loan denominator with the default rate and distinguish Q3-end subscribers from subscriptions started in Q3.